# 第六章：差异表达分析（Wilcoxon + Pseudobulk DESeq2）

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

前五篇我们做了一件事：认识你的数据。QC 去掉了坏细胞，归一化统一了量纲，整合消除了批次效应，注释给了每个细胞身份标签。我们现在有 58,735 个细胞、12 种大类、26 种细注释亚群。

但这些都是"描述型"工作——我们还没有回答任何疾病相关的问题。

从这一篇开始，我们进入机制发现阶段。第一个核心问题：肝硬化到底改变了哪些基因的表达？ 更精确地说——在每一种细胞类型中，Healthy vs Cirrhotic 之间，哪些基因显著上调或下调？

这个问题听起来简单，做起来却充满陷阱。最大的陷阱叫假阳性膨胀（pseudoreplication）——如果你用错了统计方法，你会发现"成千上万"的差异基因，但其中大部分是假的。

我们会用三种方法做 DEG 分析，对比它们的结果，帮你理解为什么 pseudobulk 是多样本 scRNA-seq 的金标准。

## 背景：单细胞 DEG 分析的三个层次

### 层次一：Marker Gene 检测

问题：每种细胞类型的特征基因是什么？

这不是"疾病 vs 对照"的比较，而是"这种细胞 vs 所有其他细胞"。目的是验证注释是否正确——如果 B 细胞的 top marker 不包含 CD79A，那注释可能有问题。

### 层次二：单细胞水平条件比较（Wilcoxon）

问题：在某种细胞类型内，Cirrhotic 和 Healthy 之间哪些基因表达不同？

最直观的做法是对每个细胞做 Wilcoxon 检验。但这里有个致命问题——每个细胞不是独立样本。同一个 donor 的细胞高度相关，把它们当成独立观测会严重低估 p 值。

### 层次三：Pseudobulk + DESeq2

问题：同上，但统计单位是 donor 而非细胞。

把同一个 donor 同一种细胞类型的所有细胞聚合成一个"假 bulk"样本，然后用成熟的 bulk RNA-seq 方法（如 DESeq2）做差异分析。这是 Squair et al., 2021 在 Nature Communications 中推荐的金标准方法。

💡 一句话总结：Wilcoxon 告诉你"有没有差异"（灵敏度高），Pseudobulk DESeq2 告诉你"这个差异是否可靠"（特异性高）。两者结合使用效果最好。

## Part A：Marker Gene 检测（Wilcoxon）

### 为什么要先做 marker 检测

在做疾病比较之前，先确认每种细胞类型的 marker 基因。这有两个目的：

验证注释质量——如果 top markers 和 canonical markers 一致，说明注释靠谱
排除 marker 基因干扰——后续的条件比较中，marker 基因可能因为细胞组成变化而"假阳性"

### 运行 Wilcoxon 检验

In [ ]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("results/04_annotated.h5ad")
print(f"输入: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"大类: {adata.obs['cell_type'].nunique()} types")

# 每个细胞类型 vs 其余所有细胞
sc.tl.rank_genes_groups(
    adata, groupby="cell_type",
    method="wilcoxon", use_raw=True,
    pts=True, key_added="wilcoxon_celltype"
)

markers_df = sc.get.rank_genes_groups_df(
    adata, group=None, key="wilcoxon_celltype"
)
markers_df.to_csv("results/05_deg/markers_celltype_wilcoxon.csv",
                   index=False)
print(f"导出: {len(markers_df)} rows")

**输出：**

> 📊 输出：
> 输入: 58735 cells × 25349 genes
> 大类: 12 types
> 导出: 304249 rows

use_raw=True 确保我们用的是 log-normalized 数据而非 scaled 数据。pts=True 会计算每个基因在目标组和背景组中的检测比例——这是判断 marker 质量的重要指标。

### 各细胞类型 Top Markers

In [ ]:
for ct in sorted(adata.obs["cell_type"].unique()):
    sub = markers_df[
        (markers_df["group"] == ct) &
        (markers_df["pvals_adj"]  1)
    ].head(5)
    genes = ", ".join(sub["names"].values)
    print(f"  {ct}: {genes}")

**输出：**

> 📊 输出：
>   B cells: CD79A, CD37, CD74, MS4A1, CD79B
>   Endothelial: CLEC4G, DNASE1L3, FCN2, FCN3, STAB2
>   HSCs/Mesenchyme: COLEC11, DCN, PTGDS, COL3A1, COL1A2
>   Hepatocytes: ALB, APOC3, APOB, APOA2, CYP2E1
>   Macrophages: C1QA, C1QB, C1QC, CD5L, MARCO
>   Monocytes: S100A8, S100A9, S100A12, VCAN, FCN1
>   NK cells: NKG7, GNLY, KLRD1, GZMB, KLRF1
>   Plasma cells: JCHAIN, MZB1, IGHA1, IGHG1, IGHG3
>   Proliferating: STMN1, TOP2A, HMGB2, KIAA0101, MKI67
>   T cells: CD3D, CD3E, TRAC, TRBC1, IL7R
>   cDC: FCER1A, CLEC10A, CD1C, PLD4, LGALS2
>   pDC: LILRA4, IRF7, CLEC4C, IL3RA, ITM2C

每个细胞类型的 top marker 都和经典知识完全一致

图 3：各细胞类型 Top 差异基因气泡图

图 3：各细胞类型 Top 差异基因气泡图

——B 细胞的 CD79A（logFC=8.26）、T 细胞的 CD3D、巨噬细胞的 C1QA/C1QB。这证实了我们的注释是可靠的。

💡 Tip：为什么 logFC 可以大于 8？

这里的 logFC 是 log2 fold change，表示该基因在目标细胞类型中的表达是背景的 2^8 = 256 倍。CD79A 在 B 细胞中检测率 94.7%，在其他细胞中仅 3.2%——这就是"绝对 marker"，几乎只在一种细胞中表达。

图 4：差异基因热图

## Part B：Healthy vs Cirrhotic（Wilcoxon，逐细胞类型）

### 思路：拆分 → 比较

对每种细胞类型，提取该类型的所有细胞，然后比较 Cirrhotic vs Healthy
的表达差异。

In [ ]:
import numpy as np

deg_results_wilcox = {}
for ct in sorted(adata.obs["cell_type"].unique()):
    mask = adata.obs["cell_type"] == ct
    n_h = ((adata.obs["condition"] == "Healthy") & mask).sum()
    n_c = ((adata.obs["condition"] == "Cirrhotic") & mask).sum()

    if n_h < 10 or n_c < 10:
        print(f"  跳过 {ct}: H={n_h}, C={n_c}")
        continue

    adata_sub = adata[mask].copy()
    sc.tl.rank_genes_groups(
        adata_sub, groupby="condition",
        reference="Healthy", method="wilcoxon",
        use_raw=True, pts=True
    )
    result = sc.get.rank_genes_groups_df(
        adata_sub, group="Cirrhotic"
    )
    result["cell_type"] = ct
    deg_results_wilcox[ct] = result

**输出：**

> 📊 输出（12 种细胞类型的 DEG 数量，|log2FC|>0.5, adj.p<0.05）：
>   B cells:         142 up,   21 down
>   Endothelial:     607 up, 1466 down
>   HSCs/Mesenchyme: 1422 up,  406 down
>   Hepatocytes:     2805 up,  165 down
>   Macrophages:     1139 up,  287 down
>   Monocytes:       1056 up,   69 down
>   NK cells:        545 up,   31 down
>   Plasma cells:     77 up,  366 down
>   Proliferating:   117 up,   47 down
>   T cells:         815 up,   95 down
>   cDC:             531 up,  100 down
>   pDC:              44 up,    6 down

数字看起来很"壮观"——肝细胞有 2805 个上调基因！但先别激动。这些数字里有大量假阳性。

### Volcano Plot 可视化

In [ ]:
import matplotlib.pyplot as plt

key_types = ["Macrophages", "T cells", "NK cells",
             "Endothelial", "Hepatocytes", "Monocytes"]
fig, axes = plt.subplots(2, 3, figsize=(21, 12))

for i, ct in enumerate(key_types):
    ax = axes.flatten()[i]
    df = deg_results_wilcox[ct].copy()
    df["-log10p"] = -np.log10(
        df["pvals_adj"].clip(lower=1e-300))
    sig = (df["pvals_adj"]  0.5)
    ax.scatter(df.loc[~sig, "logfoldchanges"],
               df.loc[~sig, "-log10p"],
               c="lightgrey", s=2, alpha=0.5)
    up = sig & (df["logfoldchanges"] > 0)
    down = sig & (df["logfoldchanges"] < 0)
    ax.scatter(df.loc[up, "logfoldchanges"],
               df.loc[up, "-log10p"],
               c="#E15759", s=3, alpha=0.6)
    ax.scatter(df.loc[down, "logfoldchanges"],
               df.loc[down, "-log10p"],
               c="#4E79A7", s=3, alpha=0.6)
    ax.set_title(f"{ct} (Cirrhotic vs Healthy)")

plt.tight_layout()
plt.savefig("results/figures/05_volcano_condition.png",
            dpi=150, bbox_inches="tight")

图 1：Wilcoxon 检验火山图——Cirrhotic vs Healthy

图 1：Wilcoxon Volcano Plot。 6 种关键细胞类型在肝硬化中的基因变化。红点为上调基因，蓝点为下调基因。

注意看 p 值的分布——很多基因的 -log10(p) 动辄超过 100，这在 bulk RNA-seq 中几乎不可能。这正是 pseudoreplication 的典型表现。

⚠️ 踩坑预警：Wilcoxon 在多样本 scRNA 中的假阳性问题

单细胞 Wilcoxon 检验把每个细胞当成独立样本。但同一个 donor 的 T 细胞之间高度相关——它们共享相同的遗传背景、取样条件、实验操作。

假设 Donor A（健康）碰巧有较高的基因 X 表达，这不是"疾病效应"而是"个体差异"。但 Wilcoxon 会把 Donor A 的几千个 T 细胞都算成独立证据，从而得出"X 基因在疾病中显著变化"的错误结论。

Squair et al., 2021 (Nature Communications) 系统性地比较了各种方法，结论很明确：对于多样本设计，pseudobulk 方法的 FDR 控制最好。单细胞水平的检验（包括 Wilcoxon、t-test、MAST）假阳性率可达 20-70%。

教训：Wilcoxon 适合做探索性分析和 marker 检测，但不能作为最终的 DEG 报告方法。

Part C：Pseudobulk DESeq2
分析

### 为什么 Pseudobulk 是金标准

Pseudobulk 的核心思想很简单：把统计单位从"细胞"变回"样本"。

具体做法：对每个 donor × cell_type 组合，把该组合内所有细胞的原始 counts 加起来，形成一个"假 bulk"样本。这样 5 个 Healthy donor + 5 个 Cirrhotic donor 就变成了 10 个 pseudobulk 样本——回到了经典的 bulk RNA-seq 差异分析框架。

然后用 DESeq2（或 edgeR）做差异分析。这些方法在 bulk RNA-seq 领域经过了十几年的验证，统计性质非常成熟。

### 构建 Pseudobulk 矩阵

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import scipy.sparse as sp

count_matrix = adata.layers["counts"]  # 原始计数

for ct in sorted(adata.obs["cell_type"].unique()):
    ct_mask = adata.obs["cell_type"] == ct
    groups = adata.obs[ct_mask].groupby(
        ["donor", "condition"])

    # 聚合：每个 donor × condition 求和
    pb_counts, pb_meta = [], []
    for (donor, cond), idx in groups.groups.items():
        if len(idx) < 10:
            continue
        cell_idx = np.where(ct_mask.values)[0]
        subset = count_matrix[cell_idx, :]
        if sp.issparse(subset):
            summed = np.array(subset.sum(axis=0)).flatten()
        else:
            summed = subset.sum(axis=0).flatten()
        pb_counts.append(summed)
        pb_meta.append({
            "donor": donor, "condition": cond,
            "cell_type": ct, "n_cells": len(idx)
        })

注意几个关键点：

必须用原始 counts——DESeq2 内部会做自己的 normalization（size factors），输入必须是未归一化的整数计数
每个 group 至少 10 个细胞——太少的话聚合后的 counts 不稳定
每个条件至少 2 个 donor——否则 DESeq2 无法估计组内方差

### 运行 DESeq2

In [ ]:
dds = DeseqDataSet(
    counts=counts_df,
    metadata=meta_df,
    design="~condition",
    refit_cooks=True, quiet=True
)
dds.deseq2()

stat_res = DeseqStats(
    dds,
    contrast=["condition", "Cirrhotic", "Healthy"],
    quiet=True
)
stat_res.summary()
result_df = stat_res.results_df.copy()

**输出：**

> 📊 输出（|log2FC|>1, padj<0.05）：
>   B cells:           0 up,   0 down
>   Endothelial:     146 up, 220 down
>   HSCs/Mesenchyme: 183 up,  31 down
>   Hepatocytes:      12 up,  24 down
>   Macrophages:     101 up, 164 down
>   Monocytes:        13 up,   2 down
>   NK cells:         10 up,   3 down
>   Plasma cells:      0 up,   0 down
>   Proliferating:     0 up,   0 down
>   T cells:          35 up,  35 down
>   cDC:              11 up,  12 down
>   pDC:               0 up,   0 down

数字变化非常剧烈！肝细胞从 Wilcoxon 的 2970 个 DEG 缩减到 DESeq2 的 36 个。这不是 DESeq2 "不灵敏"——而是 Wilcoxon 报告了太多假阳性。

⚠️ 踩坑预警：pydeseq2 的 API 细节

pydeseq2 是 DESeq2 的 Python 实现，但 API 和 R 版本略有不同：

DeseqDataSet 的 counts 参数接受 DataFrame，行是样本，列是基因——和 R 版本的转置相反
contrast 参数格式是 ["factor", "numerator", "denominator"]
输出的 results_df 中，某些列可能是 object 类型，需要 pd.to_numeric(errors="coerce") 转换

特别注意第 3 点——如果不做类型转换，后续的数值比较（如 padj < 0.05）可能报错或静默失败。

图 2：DESeq2 Pseudobulk 火山图

## 方法比较：Wilcoxon vs DESeq2

### 数量对比

细胞类型
Wilcoxon
DESeq2
交集

Endothelial
2073
366
296

HSCs/Mesenchyme
1828
214
167

Macrophages
1426
265
210

T cells
910
70
43

Hepatocytes
2970
36
17

NK cells
576
13
12

Monocytes
1125
15
12

cDC
631
23
13

B cells
163
0
0

Plasma cells
443
0
0

pDC
50
0
0

Proliferating
164
0
0

几个关键观察：

1. Wilcoxon 的结果数量远大于 DESeq2。 这印证了 pseudoreplication 的问题——当样本量只有 5 vs 5 时，Wilcoxon 利用了成千上万的细胞来"膨胀"统计效力。

2. DESeq2 的结果几乎都被 Wilcoxon 覆盖。 比如巨噬细胞：DESeq2 的 265 个 DEG 中，有 210 个也被 Wilcoxon 检测到。说明 DESeq2 找到的基因是"高置信度"子集。

3. B 细胞、浆细胞等类型在 DESeq2 中为零。 这些细胞类型在某些 donor 中数量太少（或只有单侧 donor），导致 pseudobulk 样本数不足，DESeq2 无法估计方差。

### 怎么选

推荐策略：以 DESeq2（pseudobulk）为主，Wilcoxon 为辅。

正式报告的 DEG 列表：用 DESeq2 的结果
探索性分析（比如"哪些通路可能有变化"）：用 Wilcoxon 的结果做初筛
小细胞群（DESeq2 样本不足）：退而求其次用 Wilcoxon，但要明确标注统计方法的局限

💡 Tip：为什么不直接只用 DESeq2？

因为 DESeq2 对样本量有硬性要求（每组至少 2-3 个）。在我们的数据中，pDC 只有 568 个细胞，分散在 10 个 donor 中，某些 donor 可能只有个位数的 pDC。聚合后样本量不够，DESeq2 就跑不出结果。

对这些小群体，Wilcoxon 虽然不完美，但至少能给你一个方向性的提示。

---

### 🔬 方法选择指南

🔬 方法选择指南：单细胞差异表达方法全景
方法统计模型分析单元假阳性控制推荐场景Wilcoxon ✓非参数秩和检验单细胞❌ 较差（伪重复）Marker基因检测、探索性分析DESeq2 (pseudobulk) ✓负二项GLM样本（聚合后）✅ 优秀条件比较（推荐首选）edgeR (pseudobulk)负二项GLM + 经验贝叶斯样本（聚合后）✅ 优秀条件比较（同DESeq2）MASThurdle model（zero-inflated）单细胞⚠️ 中等考虑dropout的细胞级分析t-test参数检验单细胞❌ 差❌ 通常不推荐
为什么 Pseudobulk 是条件比较的金标准

Squair et al. (2021, Nature Communications) 系统性比较了14种单细胞DE方法，核心发现：

• 细胞级方法（Wilcoxon、t-test、MAST）存在严重的伪重复问题——每个细胞被当作独立观测，但同一供体的细胞高度相关。这导致假阳性率膨胀到40~70%！
• Pseudobulk方法（DESeq2、edgeR）正确地以供体为统计单元，假阳性率接近预期的5%
• 实际验证：在我们的数据中，Wilcoxon找到的"差异基因"有很大一部分是DESeq2无法重现的——这些很可能是假阳性

决策框架

Step 1：你的目标是什么？
• 寻找细胞类型的marker基因 → Wilcoxon（每个cluster vs 其余所有）
• 比较两个条件（如 healthy vs disease）→ 进入Step 2

Step 2：你有多少生物学重复（供体/样本）？
• ≥3个/组 → Pseudobulk + DESeq2（强烈推荐）
• 2个/组 → DESeq2可以跑但统计效力很低，结果需谨慎解读
• 1个/组 → 无法做可靠的条件比较。可以用Wilcoxon做探索，但必须注明"没有生物学重复，结果仅供参考"

Step 3：DESeq2 vs edgeR？
两者在大多数情况下结果高度一致。DESeq2更流行（引用更多），edgeR对小样本略有优势。选择你更熟悉的即可。

---

## 结果解读：巨噬细胞的故事

### 巨噬细胞为什么值得关注

在 Ramachandran et al., 2019 的原文中，巨噬细胞是核心发现的主角——他们鉴定了 TREM2+CD9+ 的瘢痕相关巨噬细胞（SAMac），这群细胞在肝硬化中显著富集。

我们的 DESeq2 结果在巨噬细胞中检测到了 101 个上调基因和 164 个下调基因。上调基因中包括了多个与组织重塑和纤维化相关的基因。

### 内皮细胞：DEG 最多的细胞类型

DESeq2 在内皮细胞中检测到了最多的 DEG（146 up + 220 down = 366）。这和已知的肝硬化病理一致——肝窦内皮细胞（LSEC）在肝硬化中经历"毛细血管化"（capillarization），丢失窗孔和清道夫受体，获得 PECAM1 和 vWF 等连续性内皮标志。

这种内皮重塑影响了大量基因，所以 DEG 数量最多是合理的。

### 肝细胞：从 2970 到 36

肝细胞是 Wilcoxon 和 DESeq2 差异最大的类型（2970 vs 36）。为什么？

因为我们的数据中肝细胞主要来自 CD45- 分选的样本，不同 donor 之间的取样和消化效率差异很大——有些 donor 贡献了大量肝细胞，有些很少。Wilcoxon 把"donor 之间的异质性"和"疾病效应"混在了一起。DESeq2 通过 donor 级别的建模，成功分离了这两个来源的变异。

## 输出文件说明

本步骤生成 3 个主要输出文件：

文件名
内容
行数

markers_celltype_wilcoxon.csv
每种细胞类型的 marker genes
304,249

deg_condition_wilcoxon.csv
逐细胞类型 Wilcoxon DEG
304,249

deg_condition_deseq2.csv
逐细胞类型 Pseudobulk DESeq2 DEG
246,781

以及图片文件：

05_heatmap_markers.png — 各细胞类型 top marker 热图
05_dotplot_markers.png — Top 3 marker dotplot
05_volcano_condition.png — 6 种细胞类型的 Wilcoxon volcano
05_volcano_deseq2.png — 6 种细胞类型的 DESeq2 volcano

## 📖 与原文比较

与 Ramachandran et al., 2019 对照

原文的 DEG 分析策略和我们的第三层（pseudobulk）类似——他们对每种细胞类型做了 Healthy vs Cirrhotic 的差异分析，聚焦在巨噬细胞和内皮细胞亚群。

关键对照点：

SAMac 标志基因：原文报告的 TREM2、SPP1、GPNMB、CD9 等基因在我们的巨噬细胞 DEG 中都被检测到（Wilcoxon 显著上调），验证了这些基因的疾病关联性。

内皮细胞变化：原文描述了 LSEC 在肝硬化中丢失窗孔标志物（CLEC4G↓、STAB2↓），我们的 DESeq2 同样在 Endothelial 中检测到这些基因的下调。

统计方法差异：原文使用的是 Seurat 的 FindMarkers（基于 Wilcoxon），发表于 2019 年，当时 pseudobulk 方法还没有被广泛推荐。Squair et al., 2021 之后，pseudobulk 成为了社区共识的最佳实践。

一个值得注意的进步：我们用 pseudobulk DESeq2 得到的 DEG 列表更"保守"但更可靠。如果你在复现原文时发现 DEG 数量比原文少很多，这不是你的分析出了问题——而是你用了更好的统计方法。

## 方法论反思

### DEG 分析的"灰色地带"

1. 阈值选择是主观的。 我们用了 |log2FC|>0.5（Wilcoxon）和 |log2FC|>1（DESeq2）的阈值，但这些数字没有"正确答案"。不同的阈值会给出完全不同的 DEG 列表。在论文中，通常选择一个阈值做主分析，然后在 supplementary 中展示不同阈值的结果。

2. 多重比较校正。 我们对 12 种细胞类型分别做了 DEG 分析，但没有对"跨细胞类型"的多重比较做校正。严格来说，如果一个基因在 12 种细胞类型中都测试了，它的显著性阈值应该更严格。这在实践中很少有人做，但值得意识到。

3. 方向比数量重要。 不要陷入"我找到了 N 个 DEG"的数字游戏。更重要的问题是：这些 DEG 构成了什么样的生物学图景？哪些通路被激活了？这就是下一篇富集分析要做的事。

### 实践建议

对于你自己的项目：

如果只有 1 个样本 per 条件——只能用 Wilcoxon，但要在论文中明确声明局限性
如果有 3+ 个样本 per 条件——必须用 pseudobulk，Wilcoxon 仅作参考
如果样本量 > 10 per 条件——可以考虑 mixed model（如 MAST with random effects），但 pseudobulk 仍然是最简单最稳健的选择

## 小结

这一篇我们完成了：

✅ Marker gene 检测：12 种细胞类型的特征基因（验证注释）
✅ Wilcoxon 条件比较：逐细胞类型的 Healthy vs Cirrhotic DEG
✅ Pseudobulk DESeq2：金标准的多样本差异分析
✅ 方法比较：Wilcoxon 假阳性膨胀 vs DESeq2 保守可靠
✅ Volcano plot 可视化：6 种关键细胞类型

关键数字：

Wilcoxon DEG：最多 2970（Hepatocytes），总计数万
DESeq2 DEG：最多 366（Endothelial），数量级减少一个量级
两种方法的交集占 DESeq2 结果的 60-90%

核心教训：

多样本 scRNA-seq 必须用 pseudobulk 方法
Wilcoxon 适合探索，不适合下结论
DEG 的数量不等于质量

下一篇，我们将对这些差异基因做功能富集分析——把基因列表翻译成生物学故事。我们会用 ORA 和 GSEA 两种方法，看看肝硬化中到底激活了哪些生物学通路。